In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

# Must be called before any GPU operation — fixes "DNN library initialization failed" on WSL2
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print('GPUs found:', gpus)

model = hub.load('https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2')
total_params = sum(np.prod(v.shape) for v in model._tf_var_leaves)
print('Parameters:', f'{total_params:,}')
print('Signatures:', model.signatures)

I0000 00:00:1777258702.678684     634 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1777258705.949093     634 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4057 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:01:00.0, compute capability: 7.5


Parameters: 101,757,263
Signatures: _SignatureMap({'serving_default': <ConcreteFunction (*, inputs: TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')) -> Dict[['embedding', TensorSpec(shape=(None, 1536), dtype=tf.float32, name='embedding')], ['label', TensorSpec(shape=(None, 14795), dtype=tf.float32, name='label')], ['spectrogram', TensorSpec(shape=(None, 500, 128), dtype=tf.float32, name='spectrogram')], ['spatial_embedding', TensorSpec(shape=(None, 16, 4, 1536), dtype=tf.float32, name='spatial_embedding')]] at 0x7509679EA330>})


## Step 1 – Load a 5-second .ogg chunk → numpy array at 32 kHz

Perch v2 expects `float32` waveform, shape `(batch, 160000)`, at **32 kHz**.

In [2]:
import librosa
import numpy as np

SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: float) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
sample_path = '../data/train_audio/22930/iNat317238.ogg'  
chunk = load_chunk(sample_path, offset_sec=0.0)
print('Shape:', chunk.shape)   # (160000,)
print('dtype:', chunk.dtype)   # float32
print('Range:', chunk.min(), chunk.max())

Shape: (160000,)
dtype: float32
Range: -0.45734373 0.5143107


## Step 2 – Extract embeddings with Perch v2

`model2.infer_tf` returns a dict with keys: `embedding` (1536-d), `label` (14795 logits), `spectrogram`, `spatial_embedding`.  
We only need `embedding`.

In [3]:
infer = model.signatures['serving_default']

def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = tf.constant(waveform[np.newaxis, :])   # (1, 160000)
    out = infer(inputs=inp)
    return out['embedding'].numpy()[0]           # (1536,)

emb = extract_embedding(chunk)
print('Embedding shape:', emb.shape)  # (1536,)

I0000 00:00:1777258720.414665     765 service.cc:153] XLA service 0x75089c0883c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777258720.414703     765 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2060, Compute Capability 7.5 (Driver: 12.9.0; Runtime: 12.4.0; Toolkit: 12.5.0; DNN: 9.1.0)
I0000 00:00:1777258720.686189     765 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
E0000 00:00:1777258720.727054     765 cuda_dnn.cc:454] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.
E0000 00:00:1777258720.803328     765 cuda_dnn.cc:454] Loaded runtime CuDNN library: 

FailedPreconditionError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
<stack traces unavailable>
DNN library initialization failed. Look at the errors above for more details.
	 [[{{node StatefulPartitionedCall}}]] [Op:__inference_signature_wrapper_5524]

## Step 3 – Pre-extract all embeddings (one-time, saves to .npy)

Running Perch on every batch during training is slow. Extract once, cache to disk, then train the head on raw numpy arrays — much faster.

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from pathlib import Path

# Load your pre-built chunks dataframe (same one used for EfficientNet training)
chunks_df = pd.read_parquet('../data/chunks.parquet')   # columns: file_path, offset_sec, encoded_label

EMB_CACHE = Path('../data/perch_embeddings.npy')
LBL_CACHE = Path('../data/perch_labels.npy')

if not EMB_CACHE.exists():
    embeddings, labels = [], []
    for _, row in tqdm(chunks_df.iterrows(), total=len(chunks_df)):
        wav = load_chunk(row['file_path'], row['offset_sec'])
        emb = extract_embedding(wav)
        embeddings.append(emb)
        labels.append(row['encoded_label'])
    embeddings = np.stack(embeddings).astype(np.float32)
    labels     = np.array(labels, dtype=np.int64)
    np.save(EMB_CACHE, embeddings)
    np.save(LBL_CACHE, labels)
    print(f'Saved {len(embeddings)} embeddings → {EMB_CACHE}')
else:
    embeddings = np.load(EMB_CACHE)
    labels     = np.load(LBL_CACHE)
    print(f'Loaded {len(embeddings)} cached embeddings')

print('embeddings:', embeddings.shape)  # (N, 1536)
print('labels:    ', labels.shape)      # (N,)

## Step 4 – Train a lightweight PyTorch head

The head is just `Linear(1536 → 234)` with dropout. Perch weights stay frozen.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

NUM_CLASSES = 234
EMBED_DIM   = 1536
BATCH_SIZE  = 256
NUM_EPOCHS  = 20
LR          = 1e-3

# ---- Dataset from cached numpy arrays ----
X = torch.from_numpy(embeddings)   # (N, 1536) float32
y = torch.from_numpy(labels)       # (N,)      int64

dataset = TensorDataset(X, y)
n_train = int(0.8 * len(dataset))
n_val   = int(0.1 * len(dataset))
n_test  = len(dataset) - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---- Lightweight head ----
class PerchHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(EMBED_DIM),
            nn.Dropout(0.3),
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, NUM_CLASSES),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
head = PerchHead().to(device)
print(head)
print(f'Head parameters: {sum(p.numel() for p in head.parameters()):,}')

In [ ]:
import mlflow, mlflow.pytorch

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

mlflow.set_experiment('birdclef2026-perch-head')

best_val_loss = float('inf')
with mlflow.start_run(run_name='perch_v2_head'):
    mlflow.log_params({'epochs': NUM_EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE,
                       'embed_dim': EMBED_DIM, 'num_classes': NUM_CLASSES})

    for epoch in range(NUM_EPOCHS):
        # --- train ---
        head.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(head(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- validate ---
        head.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = head(xb)
                val_loss += criterion(preds, yb).item()
                correct  += (preds.argmax(1) == yb).sum().item()
                total    += len(yb)

        train_loss /= len(train_loader)
        val_loss   /= len(val_loader)
        val_acc     = correct / total
        scheduler.step()

        mlflow.log_metrics({'train_loss': train_loss, 'val_loss': val_loss,
                            'val_acc': val_acc}, step=epoch)
        print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            mlflow.pytorch.log_model(head, 'best_model')

print('Training complete. Best val loss:', best_val_loss)